<a href="https://colab.research.google.com/github/RayssaOliveiraa/Assistente-Virtual-com-IA-Generativa-local/blob/main/Desafio_projeto_final_(Funcionando).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🛠️ **Passo 1**: Preparação do Ambiente
Nesta célula, instalamos as bibliotecas necessárias para a interface visual e as ferramentas de conexão externa.

Streamlit: Framework que cria o chat bonitão.

Localtunnel: Ferramenta alternativa de conexão (usada como backup).

In [1]:
# Instala apenas o Streamlit via pip
!pip install streamlit -q

# Instala o Localtunnel globalmente usando o gerenciador do Node (npm)
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 112.3 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 2s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

🧠 **Passo 2**: Instalação e Ativação da IA (Ollama)
Aqui instalamos o "motor" que processa a inteligência artificial.

Baixamos o instalador oficial do Ollama.

1.   Baixamos o instalador oficial do Ollama.
2.   Iniciamos o servidor em segundo plano (background) para que ele possa responder às chamadas do chat.
3. Iniciamos o servidor em segundo plano (background) para que ele possa responder às chamadas do chat.

Fazemos o "download" do cérebro (Llama 3), que é o modelo de linguagem, mais leve para uso no colab.

In [2]:
# 1. Instala a dependência zstd (necessária para extrair o Ollama)
!sudo apt-get update && sudo apt-get install zstd -y

# 2. Instala o Ollama de fato
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Inicia o servidor em segundo plano usando o Python
import subprocess
import time
import os

print("Verificando instalação...")
# Verifica se o executável existe para evitar o erro FileNotFoundError
if os.path.exists('/usr/local/bin/ollama') or os.path.exists('/usr/bin/ollama'):
    print("✅ Ollama instalado com sucesso! Iniciando servidor...")
    subprocess.Popen(["ollama", "serve"])
    time.sleep(15) # Aguarda o servidor estabilizar

    # 4. Baixa o modelo
    !ollama pull llama3
else:
    print("❌ Erro: O Ollama não foi instalado corretamente. Tente rodar a célula novamente.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,935 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get

📝 **Passo 3**: Criação do Aplicativo (app.py)

Esta célula escreve o arquivo principal do sistema. Note que incluímos uma função de autocriação de dados (JSON/CSV) para que, caso você não faça o upload manual dos seus arquivos, o Guard Financeiro ainda consiga rodar com exemplos fictícios.

In [3]:
%%writefile app.py
import json
import pandas as pd
import requests
import streamlit as st
import os

# ============ CONFIGURAÇÃO ============
OLLAMA_URL = "http://localhost:11434/api/generate"
MODELO = "llama3"

# =========== CARREGAR DADOS ===========
# Criando dados fictícios caso as pastas não existam (para evitar erro de arquivo não encontrado)
if not os.path.exists('./data'):
    os.makedirs('./data')
    # Criando arquivos mínimos para o código não quebrar
    with open('./data/perfil_investidor.json', 'w') as f:
        json.dump({"nome": "Usuário Teste", "idade": 30, "perfil_investidor": "Moderado", "objetivo_principal": "Aposentadoria", "patrimonio_total": 50000, "reserva_emergencia_atual": 10000}, f)
    pd.DataFrame({'data': ['2023-10-01'], 'valor': [100], 'descricao': ['Teste']}).to_csv('./data/transacoes.csv', index=False)
    pd.DataFrame({'data': ['2023-09-01'], 'resumo': ['Início']}).to_csv('./data/historico_atendimento.csv', index=False)
    with open('./data/produtos_financeiros.json', 'w') as f:
        json.dump([{"nome": "CDB Exemplo", "tipo": "Renda Fixa"}], f)

perfil = json.load(open('./data/perfil_investidor.json'))
transacoes = pd.read_csv('./data/transacoes.csv')
historico = pd.read_csv('./data/historico_atendimento.csv')
produtos = json.load(open('./data/produtos_financeiros.json'))

# ============ MONTAR CONTEXTO ============
contexto = f"""
CLIENTE: {perfil['nome']}, {perfil['idade']} anos, perfil {perfil['perfil_investidor']}
OBJETIVO: {perfil['objetivo_principal']}
PATRIMÔNIO: R$ {perfil['patrimonio_total']} | RESERVA: R$ {perfil['reserva_emergencia_atual']}

TRANSAÇÕES RECENTES:
{transacoes.to_string(index=False)}

ATENDIMENTOS ANTERIORES:
{historico.to_string(index=False)}

PRODUTOS DISPONÍVEIS:
{json.dumps(produtos, indent=2, ensure_ascii=False)}

"""
# ============ SYSTEM PROMPT ============

SYSTEM_PROMPT = """Você é um educador financeiro um educador  didático e amigável e jovem chamadado " Guard Financeiro",  inteligente especializado em Finanças.

Seu objetivo é Ensinar conceitos de finanças pessoais de forma simples, usando os dados do cliente como exemplo praticos.
REGRAS:
-  Nunca recomendar investimentos especificos, apenas o seu funcionamento.
- Use os dados fornecidos para da exemplos personalizados.
- Sua linguagem é simples e clara, como se tivesse explicando a um amigo, e para quem tem pouco ou nenhum entendimento do assunto.
- Se não souber algo, admita : " Não tenha essa informações, mas posso explicar .."
- Sempre pergunte se o cliente entendeu oque você ta explicando.
- Responda  de forma direta e sucinta, com no máximo 3 paragrafos.
- JAMAIS responda a perguntas fora do tema ensino de finanças pessoais.
  Quando ocorrer, responda lembrando o seu papel de educador financeiro;"""

def perguntar(msg):
    prompt = f"{SYSTEM_PROMPT}\n\nCONTEXTO DO CLIENTE:\n{contexto}\n\nPergunta: {msg}"
    try:
        r = requests.post(OLLAMA_URL, json={"model": MODELO, "prompt": prompt, "stream": False})
        return r.json()['response']
    except Exception as e:
        return f"Erro ao conectar com o Ollama: {e}"

# =========== INTERFACE ===========
st.title("🎓 Olá sou Guard Financeiro, seu educador financeiro!")

if pergunta := st.chat_input("Sua dúvida sobre finanças..."):
    st.chat_message("user").write(pergunta)
    with st.spinner("Pensando..."):
        resposta = perguntar(pergunta)
        st.chat_message("assistant").write(resposta)

Writing app.py


🔍 **Passo 4**: Diagnóstico de Arquivos. (**Não obrigatorório, apenas usado para verificação**)

Célula de segurança para garantir que todos os arquivos necessários para o Guard Financeiro estão presentes no disco rígido do Colab.

In [ ]:
import os

# Verifica se a pasta data existe
if os.path.exists('./data'):
    print("✅ Pasta 'data' encontrada!")
    print("Arquivos dentro dela:", os.listdir('./data'))
else:
    print("❌ Pasta 'data' NÃO encontrada!")

# Verifica se o arquivo app.py existe
if os.path.exists('app.py'):
    print("✅ Arquivo 'app.py' encontrado!")
else:
    print("❌ Arquivo 'app.py' NÃO encontrado!")

✅ Pasta 'data' encontrada!
Arquivos dentro dela: ['transacoes.csv', 'perfil_investidor.json', 'historico_atendimento.csv', 'produtos_financeiros.json']
✅ Arquivo 'app.py' encontrado!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

⚡**Passo 5** : Teste de Resposta da IA. (Não obrigatorório, apenas usado para verificação)

  Antes de abrir o chat, verificamos se o servidor da IA (Ollama) está "acordado" e pronto para receber perguntas.

In [ ]:
import requests
try:
    r = requests.get("http://127.0.0.1:11434/")
    print("✅ Servidor Ollama está ATIVO e respondendo!")
except:
    print("❌ Servidor Ollama ainda está desligado.")

✅ Servidor Ollama está ATIVO e respondendo!


🚀 **Passo 6**: Lançamento com Túnel Seguro (Ngrok)

Esta é a última etapa. Ela gera o link externo para você acessar o app. Usamos o Ngrok por ser o método mais estável para evitar que o ***campo de chat fique travado***.

In [ ]:
# Instala o Ngrok
!pip install pyngrok -q
from pyngrok import ngrok

# --- CONFIGURAÇÃO ---
# Cole seu token entre as aspas abaixo:
NGROK_TOKEN = "Cole seu token"
ngrok.set_auth_token(NGROK_TOKEN)

# Mata túneis e processos antigos para não dar conflito
ngrok.kill()
!fuser -k 8501/tcp

# Abre o túnel
public_url = ngrok.connect(8501, proto="http")
print(f"\n🚀 APP ONLINE! Acesse aqui: {public_url}")
print("--------------------------------------------------")

# Roda o Streamlit com flags de estabilidade
!streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false


🚀 APP ONLINE! Acesse aqui: NgrokTunnel: "https://superrighteous-cora-patiently.ngrok-free.dev" -> "http://localhost:8501"
--------------------------------------------------



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.229.129.111:8501



🌐 Entendendo o Ngrok no Google Colab

O que é o Ngrok?
Imagine que o Google Colab é uma sala trancada por dentro. Você criou seu app (Streamlit) lá dentro, mas ninguém consegue entrar para ver, ***que foi oque aconteceu nos testes*** . Logo o Ngrok funciona como um túnel seguro: ele cria uma passagem direta da internet para dentro dessa sala, gerando um link exclusivo para você acessar seu projeto de qualquer lugar, a solução viavel para nosso assistente funcinar.


🛠️ O que você precisa fazer (Passo a Passo):

Para que o túnel funcione corretamente, o usuário deve seguir este fluxo:

1. Obter o Token de Acesso: * Vá ao site dashboard.ngrok.com e crie uma conta gratuita.

2. Copie o seu Authtoken (uma sequência longa de letras e números).

3. Configurar no Código: * Cole esse token na variável NGROK_TOKEN dentro da célula de execução.

4. Limpeza de Sessão: Sempre que rodar o código novamente, o script irá encerrar túneis antigos (ngrok.kill()) para evitar erros de "múltiplas conexões", que é o limite da conta gratuita.

    Acesso ao Link: * Ao rodar a célula, o Ngrok gerará um link azul (ex: **https://abcd-123.ngrok-free.app**).

* Ao clicar, você verá uma tela de aviso do Ngrok; basta clicar no botão "Visit
Site" para entrar no seu Guard Financeiro.

**Por que usamos o Ngrok aqui?**

* Estabilidade: Diferente de outros métodos, ele não deixa o campo de digitar (chat input) travar.

* Segurança: Toda a comunicação é criptografada.

* Velocidade: É a forma mais rápida de conectar o Streamlit ao seu navegador sem precisar configurar redes complexas.